In [179]:
import numpy as np

x = np.arange(-5, 5.1, 0.5)
y = np.arange(-5, 5.1, 0.5)
X,Y = np.meshgrid(x,y)
features = np.array([X.flatten(),Y.flatten()]).T
labels = (10 * features[:,0] + 5 * features[:,1] - 3 + np.random.normal(size=features.shape[0], scale=0.1))[:,np.newaxis]

train_indices = np.arange(int(len(features) * 0.7))
test_indices = np.arange(int(len(features) * 0.7)+1, len(features))

In [180]:
from sklearn.linear_model import LinearRegression

linear = LinearRegression().fit(features[train_indices], labels[train_indices])
prediction = linear.predict(features[test_indices])
error = np.linalg.norm(prediction - labels[test_indices]) / len(test_indices)

error, linear.coef_, linear.intercept_

(np.float64(0.009075209901492399),
 array([[9.99827911, 4.99599076]]),
 array([-3.01075817]))

In [190]:
import torch

class LinearNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = torch.nn.Flatten()
        self.net = torch.nn.Linear(2,1)

    def forward(self, x):
        x = self.flatten(x)
        logits = self.net(x)
        return logits

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"    
network_model = LinearNetwork().to(device)

loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.SGD(network_model.parameters(), lr=1e-3)

batch_size = 16
epochs = 500

for epoch in range(epochs):
    network_model.train()
    
    np.random.shuffle(train_indices)
    for batch_start in np.arange(0, len(train_indices)-batch_size-1, batch_size):
        indices = train_indices[batch_start:batch_start+batch_size]
        X = torch.tensor(features[indices], dtype=torch.float32).to(device)
        y = torch.tensor(labels[indices], dtype=torch.float32).to(device)
        
        pred = network_model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
    network_model.eval()
    test_loss, correct = 0.0, 0.0
    with torch.no_grad():
        X = torch.tensor(features[train_indices], dtype=torch.float32).to(device)
        y = torch.tensor(labels[train_indices], dtype=torch.float32).to(device)

        pred = network_model(X)
        test_loss += loss_fn(pred, y).item()

    test_loss /= len(train_indices)
    if (epoch % 30 == 0):
        print(f"Test Error: Avg loss: {test_loss:>8f}")

with torch.no_grad():
    prediction = network_model(torch.tensor(features[test_indices], dtype=torch.float32).to(device))
error = np.linalg.norm((prediction - labels[test_indices]).flatten()) / len(test_indices)

error, list(network_model.parameters())

Test Error: Avg loss: 2.091881
Test Error: Avg loss: 0.002259
Test Error: Avg loss: 0.000589
Test Error: Avg loss: 0.000172
Test Error: Avg loss: 0.000067
Test Error: Avg loss: 0.000041
Test Error: Avg loss: 0.000034
Test Error: Avg loss: 0.000032
Test Error: Avg loss: 0.000032
Test Error: Avg loss: 0.000032
Test Error: Avg loss: 0.000032
Test Error: Avg loss: 0.000032
Test Error: Avg loss: 0.000032
Test Error: Avg loss: 0.000032
Test Error: Avg loss: 0.000032
Test Error: Avg loss: 0.000032
Test Error: Avg loss: 0.000032


/tmp/ipykernel_43545/3093055196.py:54: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  error = np.linalg.norm((prediction - labels[test_indices]).flatten()) / len(test_indices)


(np.float64(0.009068509348897188),
 [Parameter containing:
  tensor([[9.9984, 4.9960]], requires_grad=True),
  Parameter containing:
  tensor([-3.0107], requires_grad=True)])

In [191]:
class LinearRegressionManual():
    def __init__(self, num_inputs):
        self.w = torch.normal(0, 1, size=(num_inputs,1),requires_grad=True)
        self.b = torch.normal(0, 1, size=(1,1), requires_grad=True)

    def forward(self, X):
        return X @ self.w + self.b

    def loss(self, y_hat, y):
        l = (y_hat - y) ** 2 / 2
        return l.mean()

    def __call__(self, X):
        return self.forward(X)

class SGD:
    """Minibatch stochastic gradient descent."""
    def __init__(self, params, lr):
        self.params = params
        self.lr = lr

    def step(self):
        for param in self.params:
            param -= self.lr * param.grad

    def zero_grad(self):
        for param in self.params:
            if param.grad is not None:
                param.grad.zero_()


manual_regression_model = LinearRegressionManual(2)
optimizer = SGD([manual_regression_model.w, manual_regression_model.b], 1e-3)

batch_size = 16
epochs = 500

for epoch in range(epochs):
    np.random.shuffle(train_indices)
    for batch_start in np.arange(0, len(train_indices)-batch_size-1, batch_size):
        indices = train_indices[batch_start:batch_start+batch_size]
        X = torch.tensor(features[indices], dtype=torch.float32).to(device)
        y = torch.tensor(labels[indices], dtype=torch.float32).to(device)

        pred = manual_regression_model(X)
        loss = manual_regression_model.loss(pred, y)

        optimizer.zero_grad()
        with torch.no_grad():
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
    
    test_loss, correct = 0.0, 0.0
    with torch.no_grad():
        X = torch.tensor(features[train_indices], dtype=torch.float32).to(device)
        y = torch.tensor(labels[train_indices], dtype=torch.float32).to(device)

        pred = manual_regression_model(X)
        test_loss += loss_fn(pred, y).item()

    test_loss /= len(train_indices)
    if (epoch % 30 == 0):
        print(f"Test Error: Avg loss: {test_loss:>8f}")

with torch.no_grad():
    prediction = manual_regression_model(torch.tensor(features[test_indices], dtype=torch.float32).to(device))
error = np.linalg.norm((prediction - labels[test_indices]).flatten()) / len(test_indices)

error, [manual_regression_model.w, manual_regression_model.b]

Test Error: Avg loss: 3.104921
Test Error: Avg loss: 0.001028
Test Error: Avg loss: 0.000427
Test Error: Avg loss: 0.000230
Test Error: Avg loss: 0.000131
Test Error: Avg loss: 0.000081
Test Error: Avg loss: 0.000057
Test Error: Avg loss: 0.000044
Test Error: Avg loss: 0.000038
Test Error: Avg loss: 0.000035
Test Error: Avg loss: 0.000033
Test Error: Avg loss: 0.000033
Test Error: Avg loss: 0.000032
Test Error: Avg loss: 0.000032
Test Error: Avg loss: 0.000032
Test Error: Avg loss: 0.000032
Test Error: Avg loss: 0.000032


/tmp/ipykernel_43545/2344746834.py:68: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  error = np.linalg.norm((prediction - labels[test_indices]).flatten()) / len(test_indices)


(np.float64(0.008993760069964845),
 [tensor([[9.9983],
          [4.9966]], requires_grad=True),
  tensor([[-3.0080]], requires_grad=True)])